In [7]:
import importlib.util
import sys
from pathlib import Path

SCHEMAS_DIR = Path("/home/hello/Projects/Statements/code/moltie/schemas")
assert SCHEMAS_DIR.exists(), f"Missing schemas dir: {SCHEMAS_DIR}"

PKG = "schemas"  # pretend package name

# Create an empty package module object for `schemas`
if PKG not in sys.modules:
    pkg_spec = importlib.util.spec_from_loader(PKG, loader=None)
    pkg_mod = importlib.util.module_from_spec(pkg_spec)
    pkg_mod.__path__ = [str(SCHEMAS_DIR)]  # mark as package
    sys.modules[PKG] = pkg_mod

def import_as_pkg(mod_name: str, file_path: Path):
    full_name = f"{PKG}.{mod_name}"
    spec = importlib.util.spec_from_file_location(full_name, str(file_path))
    assert spec and spec.loader, f"Cannot load spec for {file_path}"
    mod = importlib.util.module_from_spec(spec)
    sys.modules[full_name] = mod
    spec.loader.exec_module(mod)
    return mod

query_object  = import_as_pkg("query_object",  SCHEMAS_DIR / "query_object.py")
verdict       = import_as_pkg("verdict",       SCHEMAS_DIR / "verdict.py")
negative_exit = import_as_pkg("negative_exit", SCHEMAS_DIR / "negative_exit.py")
run_config    = import_as_pkg("run_config",    SCHEMAS_DIR / "run_config.py")

QueryObject  = query_object.QueryObject
AtomQuery    = query_object.AtomQuery
Verdict      = verdict.Verdict
Anchor       = verdict.Anchor
NegativeExit = negative_exit.NegativeExit
RunConfig    = run_config.RunConfig

print("✅ Loaded schemas as an in-memory package namespace")


✅ Loaded schemas as an in-memory package namespace


In [8]:
from pathlib import Path
import PyPDF2

appeal_dir = Path("/media/hello/Vault/Tribunals/EAT_Appeals/")
appeal_name = "Z_v_A_UKEAT_0203_13_SM_.pdf" # <-- change this

assert appeal_dir.exists(), f"Missing dir: {appeal_dir}"


PDF_PATH = appeal_dir / appeal_name  # <-- change this
assert PDF_PATH.exists(), f"Missing PDF: {PDF_PATH}"

def pdf_to_text(path: Path) -> str:
    out = []
    with path.open("rb") as f:
        reader = PyPDF2.PdfReader(f)
        for i, page in enumerate(reader.pages):
            txt = page.extract_text() or ""
            out.append(txt)
    return "\n".join(out)

doc_text = pdf_to_text(PDF_PATH)
print("PDF chars:", len(doc_text))
print("Preview:\n", doc_text[:1200])


PDF chars: 52945
Preview:
  Copyright 2013  Appeal No.  UKEAT /0203/13/SM  
    & UKEAT/0380/13/SM  
 
 
EMPLOYMENT APPEAL TRIBUNAL  
FLEETBANK HOUSE, 2 -6 SALISBURY SQUARE, LONDON, EC4Y 8JX  
 
 
 At the Tribunal  
 on 12th November 2013  
    Judgment handed down o n 9th December 2013  
 
 
Before  
THE HONOURABLE MR JUSTI CE LANGSTAFF  (PRESIDENT)  
MR I EZEKIEL  
MR H SINGH  
 
  
UKEAT/0203/13/SM  
 
Z  APPELLANT  
 
 
 
 
 
A RESPONDENT  
 
  
UKEAT/0380/13/SM  
 
A APPELLANT  
 
 
 
 
 
Z 
 RESPONDENT  
 
 
 
JUDGMENT  
 
 
UKEAT/0203/13/SM  
UKEAT/0308/13/SM  
   
 
 
 
 
 
 
 APPEARANCES  
 
 
 
 
 
For the Appellant  
(in UKEAT/0203/13/SM  
and the Respondent in  
UKEAT/0380/13/SM)  
 MR ANDREW WATSON  
(Representative)  
Instructed by:  
Free Representation Unit  
Ground Floor  
60 Gray's Inn Road  
London  
WC1X 8LU  
 
For the Respondent  
(in UKEAT/0203/13/SM  
and the Appellant in  
UKEAT/0380/13/SM)  
 MR BRUCE GARDINER  
 (of Counsel)  
Instructed by:  
Legal Services

In [9]:
import re
from dataclasses import dataclass
from typing import List, Tuple

QUERY_TEXT = """
appeal treated as final
predetermined
open-minded review
allegation presented as established conclusion
unauthorised disclosure
"""  # <-- paste your query here (WS-derived, or X indicators)

# ---- simple paragraph splitter ----
PARA_SPLIT = re.compile(r"\n\s*\n+")
WS_RE = re.compile(r"[ \t]+")

def clean_para(s: str) -> str:
    s = s.strip()
    s = WS_RE.sub(" ", s)
    return s.strip()

@dataclass(frozen=True)
class Para:
    para_id: str
    text: str

def to_paras(text: str) -> List[Para]:
    parts = PARA_SPLIT.split(text or "")
    paras = []
    n = 0
    for p in parts:
        p = clean_para(p)
        if len(p) < 20:
            continue
        n += 1
        paras.append(Para(para_id=f"p{n:05d}", text=p))
    return paras

paras = to_paras(doc_text)
print("Paras:", len(paras))

# ---- dumb keyword scoring ----
terms = [t.strip().lower() for t in QUERY_TEXT.splitlines() if t.strip()]
def score_para(p: Para) -> int:
    t = p.text.lower()
    return sum(1 for term in terms if term in t)

scored: List[Tuple[int, Para]] = [(score_para(p), p) for p in paras]
scored = [(s,p) for s,p in scored if s > 0]
scored.sort(key=lambda x: x[0], reverse=True)

print("Matched paras:", len(scored))
print("\n=== Top 10 matches ===")
for s, p in scored[:10]:
    print(f"\n[{p.para_id}] score={s}")
    print(p.text[:800], "..." if len(p.text) > 800 else "")


Paras: 89
Matched paras: 0

=== Top 10 matches ===
